In [0]:
import dataiku
import pandas as pd

In [0]:
# Replace with the name of the S3 connection you are migrating away from.
old_connection_name = "personal_AWS_S3"
# Replace with the name of the target S3 connection to migrate datasets to.
new_connection_name = "dataiku-managed-storage"


In [0]:
# Initialize the Dataiku API client
client = dataiku.api_client()

# Get the current project
project = client.get_default_project()

# List all datasets in the current project
unique_dataset_names = set(dataset['name'] for dataset in project.list_datasets())

In [0]:
def get_dataset_configs():
    """Return a DataFrame of dataset type, connection, and path info for all datasets in the current project."""
    extracted_data = [
        {
            'type': row.get('type'),
            'connection': row.get('params', {}).get('connection'),
            'name': row.get('name'),
            'table': row.get('params', {}).get('table'),
            'catalog': row.get('params', {}).get('catalog'),
            'schema': row.get('params', {}).get('schema'),
            'path':  row.get('params', {}).get('path'),
        }
        for row in project.list_datasets()
    ]
    return pd.DataFrame(extracted_data).sort_values(by=['type', 'connection', 'name'])


In [0]:
# display the BEFORE
get_dataset_configs()

In [0]:
for dataset_name in unique_dataset_names:
    settings = project.get_dataset(dataset_name).get_settings()
    if settings.get_raw().get('params', {}).get('connection') == old_connection_name:
        # Update to the new S3 connection
        settings.set_connection_and_path(new_connection_name, settings.get_raw_params()['path'])
        settings.save()
        print(f"Dataset {dataset_name} updated to use connection: {new_connection_name}")

In [0]:
# display the AFTER for comparison
get_dataset_configs()